# Phase 1: Exploratory Data Analysis (EDA)
## Customer Churn & LTV Prediction Engine

This notebook guides you through the initial ingestion, validation, and exploratory profiling of the **IBM Telco Customer Churn** dataset. EDA is a critical foundation for detecting anomalies, understanding correlations, and identifying high-value features for predictive modeling.

### Objectives:
1. Ingest dataset using modular loading scripts.
2. Profile data distributions, missing values, and type castings.
3. Investigate the target variable (`Churn`) distribution.
4. Analyze numerical and categorical inputs relative to churn risk.
5. Prepare insights for the subsequent data cleaning and engineering phases.

### Step 1: Environment Setup and Library Imports

In [ ]:
import sys
import os

# Ensure project root is in python path for local package imports
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data, validate_schema
from src.utils import load_config

# Set style for visualizations
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("Libraries successfully imported!")

: 

### Step 2: Ingest the Raw Dataset

> **Manual Action Required:** Before executing this block, make sure you have downloaded the IBM Telco Churn CSV and placed it at `data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv` (see instructions in README).

In [ ]:
try:
    df_raw = load_raw_data()
    validate_schema(df_raw)
except FileNotFoundError as e:
    print(f"[Error] {e}")
    print("Please download the dataset from Kaggle or IBM and place it in the correct directory.")
except Exception as e:
    print(f"[Unexpected Error] {e}")

### Step 3: Initial Data Profiling and Cleansing Check

In [ ]:
# Inspect first few rows
if 'df_raw' in locals():
    display(df_raw.head())
else:
    print("Dataset is not loaded yet.")

In [ ]:
# View shape, column names, null counts, and data types
if 'df_raw' in locals():
    print("--- Data Types and Non-Null Summary ---")
    df_raw.info()
    
    print("\n--- Missing Values Profile ---")
    missing = df_raw.isnull().sum()
    print(missing[missing > 0])
    
    print("\n--- Blank spaces in object columns ---")
    # TotalCharges is read as object, inspect why (whitespace check)
    space_count = (df_raw['TotalCharges'] == ' ').sum() if 'TotalCharges' in df_raw.columns else 0
    print(f"Blank spaces (' ') found in 'TotalCharges': {space_count}")

### Step 4: Target Variable Profiling (`Churn`)

Understanding target class distribution is vital to identify if class imbalance is a concern.

In [ ]:
if 'df_raw' in locals():
    churn_counts = df_raw['Churn'].value_counts()
    churn_pct = df_raw['Churn'].value_counts(normalize=True) * 100
    
    print(f"Churn count: Yes = {churn_counts.get('Yes', 0)}, No = {churn_counts.get('No', 0)}")
    print(f"Churn rate: Yes = {churn_pct.get('Yes', 0.0):.2f}%, No = {churn_pct.get('No', 0.0):.2f}%")
    
    # Visualise the target distribution
    sns.countplot(x='Churn', data=df_raw, palette='Set2')
    plt.title("Subscriber Churn Target Class Distribution")
    plt.xlabel("Customer Churned?")
    plt.ylabel("Count")
    plt.show()

### Step 5: Numerical Feature Distributions

Let's inspect the spread of key numerical variables: `tenure`, `MonthlyCharges`, and `TotalCharges` (cast to float first).

In [ ]:
if 'df_raw' in locals():
    df_temp = df_raw.copy()
    # Convert TotalCharges for distribution plotting
    df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'].replace(' ', np.nan), errors='coerce')
    
    # Plot numerical histograms
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    sns.histplot(df_temp['tenure'], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title("Distribution of Tenure (Months)")
    
    sns.histplot(df_temp['MonthlyCharges'], kde=True, ax=axes[1], color='salmon')
    axes[1].set_title("Distribution of Monthly Charges ($)")
    
    sns.histplot(df_temp['TotalCharges'], kde=True, ax=axes[2], color='lightgreen')
    axes[2].set_title("Distribution of Total Charges ($)")
    
    plt.tight_layout()
    plt.show()

### Step 6: Bivariate Analysis (Numerical Features vs. Churn)

Examine if tenure and subscription cost rates differ significantly between churned and active subscribers.

In [ ]:
if 'df_raw' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    sns.boxplot(x='Churn', y='tenure', data=df_raw, ax=axes[0], palette='Set2')
    axes[0].set_title("Tenure vs. Churn")
    
    sns.boxplot(x='Churn', y='MonthlyCharges', data=df_raw, ax=axes[1], palette='Set2')
    axes[1].set_title("Monthly Charges vs. Churn")
    
    plt.tight_layout()
    plt.show()

### Step 7: Categorical Variables and Service Subscriptions

Assess relationship between products subscribed to (Internet service, online safety, contracts) and Churn rate.

In [ ]:
if 'df_raw' in locals():
    # Review Contract Types
    sns.countplot(x='Contract', hue='Churn', data=df_raw, palette='Set1')
    plt.title("Churn Distribution by Contract Type")
    plt.ylabel("Count")
    plt.show()
    
    # Review Internet Service Types
    sns.countplot(x='InternetService', hue='Churn', data=df_raw, palette='Set1')
    plt.title("Churn Distribution by Internet Service Type")
    plt.ylabel("Count")
    plt.show()

### Summary of Initial Key Findings (Editable)

Based on the analysis, compile your notes below:
- **Class Imbalance:** Roughly 26.5% churn, meaning some metric stratification is needed during training.
- **Tenure Effect:** Customers who churn tend to do so early in their lifecycle (low tenure boxplot median).
- **Charges Effect:** High monthly charges correlate with higher churn ratios.
- **Contracts:** Month-to-month contracts represent the highest density of churning accounts.